# HW1 — Binary entropy and the Hellman–Raviv self-consistency

**Reading.** `handout.md` §Q1–Q5 in this directory.

**Doctrine.** Each concept appears in an 8-cell rhythm:
concept (md) → demo (code) → distinguish (md) → replicate-and-vary (code) → gate (md) → student-TODO (code) → assert-gate (code) → reflection (code).

**Goal.** Implement `hbin`, `bayes_error`, and `hr_violations`; verify HR on a 10 001-point grid; design three mutations that break HR and one negative-control mutation that keeps it; log every claim with a calibrated confidence level.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw1', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


## Q1 — Binary entropy

**Concept.** $H_{\mathrm{bin}}(p) = -p \log_2 p - (1-p)\log_2(1-p)$, in bits, with $0 \log_2 0 := 0$. Four properties to demonstrate numerically before proving in the writeup:

(a) $H_\mathrm{bin}(0) = H_\mathrm{bin}(1) = 0$.
(b) $H_\mathrm{bin}(p) = H_\mathrm{bin}(1-p)$ — symmetry.
(c) $H_\mathrm{bin}(1/2) = 1$ bit.
(d) Strictly concave on $(0, 1)$.

In [ ]:
from math import log2

def hbin(p: float) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — concavity vs unimodality.** $H_\mathrm{bin}$ is both concave *and* unimodal with peak at $1/2$; unimodality alone would not imply Hellman–Raviv. We overlay the secant chord from $(0.1, H(0.1))$ to $(0.9, H(0.9))$ to make concavity visible.

In [ ]:
grid = np.linspace(0, 1, 1001)
vals = np.array([hbin(p) for p in grid])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(grid, vals, lw=2, label=r'$H_{\mathrm{bin}}(p)$')
xs = np.array([0.1, 0.9]); ys = np.array([hbin(0.1), hbin(0.9)])
ax.plot(xs, ys, 'k--', lw=1, label='secant chord')
ax.set_xlabel('p'); ax.set_ylabel('bits'); ax.legend()
ax.set_title('Q1 — binary entropy and a concavity chord')
plt.tight_layout(); plt.show()
print(f'hbin(0)={hbin(0)} hbin(1)={hbin(1)} hbin(0.5)={hbin(0.5)}')
print(f'max symmetry defect = {np.max(np.abs(vals - vals[::-1])):.2e}')


**Gate Q1.** Four invariants — each checked numerically below. If any assertion fires, the printed message names the failing property.

In [ ]:
# Gate: hbin satisfies the four named properties.
assert hbin(0) == 0 and hbin(1) == 0, 'Q1.a: hbin(0)=hbin(1)=0 violated'
assert abs(hbin(0.5) - 1.0) < 1e-12, 'Q1.c: hbin(1/2)=1 violated'
import random
random.seed(0)
for _ in range(100):
    p = random.uniform(1e-6, 1 - 1e-6)
    assert abs(hbin(p) - hbin(1 - p)) < 1e-12, f'Q1.b: asymmetry at p={p}'
assert abs(hbin(0.1) - 0.46899559358928133) < 1e-12, 'Q1: hbin(0.1) value mismatch'
print('[GATE OK] Q1: hbin satisfies (a)(b)(c) and matches reference value at p=0.1')


In [ ]:
reflect.log('Q1_hbin',
            'Hbin satisfies the four basic properties; hbin(0.1)=0.4689955935...',
            'HIGH')


## Q2 — Bayes error of a single coin

**Concept.** For $Y \sim \mathrm{Bernoulli}(p)$, the minimum-error deterministic classifier predicts the majority class and incurs $\varepsilon(p) = \min(p, 1-p)$. Randomised classifiers cannot beat this (Jensen on 0/1 loss).

In [ ]:
def bayes_error(p: float) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — `bayes_error` vs `hbin`.** Both vanish at the endpoints and peak at $1/2$, but $\varepsilon$ is *piecewise linear* (kinked at $1/2$) whereas $H_\mathrm{bin}$ is *smooth and concave*. The slack of HR rides on this distinction.

In [ ]:
ps = np.linspace(0, 1, 501)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ps, [bayes_error(p) for p in ps], label=r'$\varepsilon(p)$ (piecewise linear)')
ax.plot(ps, [hbin(p) for p in ps], label=r'$H_\mathrm{bin}(p)$ (smooth)')
ax.axvline(0.5, color='grey', lw=0.5, ls=':')
ax.legend(); ax.set_xlabel('p'); ax.set_title('Q2 — Bayes error vs binary entropy')
plt.tight_layout(); plt.show()
print('ε grid (p=0..1 in 0.1 steps):', np.round([bayes_error(p) for p in np.linspace(0,1,11)], 4))


**Gate Q2.** $\varepsilon(p) \le 1/2$ everywhere; $\varepsilon(p) = \varepsilon(1-p)$ (symmetry); spot-check $\varepsilon(0.3) = \varepsilon(0.7) = 0.3$.

In [ ]:
# Gate: bayes_error is bounded by 1/2 and symmetric.
for p in np.linspace(0, 1, 101):
    assert bayes_error(p) <= 0.5 + 1e-12
    assert abs(bayes_error(p) - bayes_error(1 - p)) < 1e-12
assert abs(bayes_error(0.3) - 0.3) < 1e-12
assert abs(bayes_error(0.7) - 0.3) < 1e-12
print('[GATE OK] Q2: bayes_error symmetric, bounded by 1/2, ε(0.3)=ε(0.7)=0.3')


In [ ]:
reflect.log('Q2_bayes',
            'Bayes error of Bernoulli(p) is min(p,1-p); symmetric and bounded by 1/2',
            'HIGH')


## Q3 — Verify Hellman–Raviv on a 10 001-point grid

**Concept.** $\varepsilon(p) \le \tfrac{1}{2} H_\mathrm{bin}(\varepsilon(p))$ for all $p \in [0, 1]$. We compute LHS, RHS, slack on a grid, count violations, and locate the maximum slack.

**Calibration tag the paper uses for $w^*$:** $w^* \approx 0.1610$, attained at $\varepsilon^* = 1/5$, i.e. at $p^* = 1/5$ or $4/5$.

In [ ]:
def hr_violations(grid: np.ndarray, atol: float = 1e-12) -> dict:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — single-coin slack vs partition $w^*$.** Q3 measures the slack of HR on a *single coin*; the paper's $w^* \approx 0.1610$ is slack at the maximum over fixed-$H$ *partitions*. They coincide numerically because the single-coin sweep happens to traverse the same upper envelope.

In [ ]:
grid = np.linspace(0, 1, 10_001)
out = hr_violations(grid)
print(out)
ps = np.linspace(0, 1, 501)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, [bayes_error(p) for p in ps], label=r'LHS: $\varepsilon(p)$', lw=2)
ax.plot(ps, [0.5 * hbin(bayes_error(p)) for p in ps],
        label=r'RHS: $\frac{1}{2} H_\mathrm{bin}(\varepsilon(p))$', lw=2)
ax.axvline(out['argmax_p'], color='red', lw=1, ls='--',
           label=f"argmax_p={out['argmax_p']:.3f}")
ax.set_xlabel('p'); ax.set_ylabel('bits / fraction'); ax.legend()
ax.set_title('Q3 — Hellman–Raviv envelope on a Bernoulli coin')
plt.tight_layout()
_plots = _PROJECTS / 'psets' / 'hw1' / 'plots'; _plots.mkdir(exist_ok=True)
fig.savefig(_plots / 'hw1_q3_envelope.png', dpi=120)
plt.show()
print(f"saved {_plots / 'hw1_q3_envelope.png'}")


**Gate Q3.** Rubric: `violations == 0`, `0.10 < max_slack < 0.20`, and the *Bayes error* at the argmax is approximately $1/5$ (slack is symmetric about $p = 1/2$, so `argmax_p` itself may be either $\approx 1/5$ or $\approx 4/5$). Tighter spot-check: `max_slack ≈ 0.1610`.

In [ ]:
# Gate: HR holds on the grid; max slack matches w* = 0.1610 at ε* = 1/5.
assert out['violations'] == 0, f"Q3: HR violated {out['violations']} times — bug"
assert 0.10 < out['max_slack'] < 0.20, f"Q3: max_slack={out['max_slack']} out of expected range"
_eps_star = bayes_error(out['argmax_p'])
assert abs(_eps_star - 0.2) < 2e-3, f"Q3: ε(argmax_p)={_eps_star} ≠ 1/5"
assert abs(out['max_slack'] - 0.1610) < 1e-3, f"Q3: max_slack={out['max_slack']} ≠ w*"
print(f"[GATE OK] Q3: violations=0, max_slack={out['max_slack']:.4f}, ε(argmax_p)={_eps_star:.4f}")


In [ ]:
reflect.log('Q3_hr_grid',
            f"HR holds on a 10001-point grid; max_slack={out['max_slack']:.4f} at p={out['argmax_p']:.3f}",
            'HIGH')


## Q4 — Mutate → fail → revert

**Concept.** A verifier you cannot break with a *deliberately wrong* predicate is a rubber stamp. We design three single-line mutations of the HR predicate:

- **A** — shrink the $\tfrac{1}{2}$ factor to $0.4$. (Breaks   Hellman–Raviv's slack margin → SHOULD violate.)
- **B** — replace $\varepsilon$ with $\varepsilon^2$ inside   $H_\mathrm{bin}$. (Breaks the proof's curvature step →   SHOULD violate.)
- **C** — equality case: set LHS := RHS. (The verifier must   *not* fire false positives on the boundary → SHOULD hold.)

In [ ]:
def mutation_A(grid: np.ndarray, atol: float = 1e-12) -> dict:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — falsification, not confirmation.** A test suite that only contains positive examples ("this should pass") doesn't prove the test machinery works. Mutation C is the *negative control*: we deliberately set LHS = RHS and confirm the predicate `lhs > rhs + atol` does *not* trip.

In [ ]:
grid = np.linspace(0, 1, 10_001)
A = mutation_A(grid); B = mutation_B(grid); C = mutation_C(grid)
print('A (should violate):', A)
print('B (should violate):', B)
print('C (negative ctrl, should hold):', C)


**Gate Q4.** A and B fire; C does not. Slack at A's worst-case should be at least $0.01$; B's at least $0.05$.

In [ ]:
# Gate: mutations behave as designed.
assert A['violations'] > 0, f'Q4 mutation_A should violate; got {A}'
assert B['violations'] > 0, f'Q4 mutation_B should violate; got {B}'
assert C['violations'] == 0, f'Q4 mutation_C must NOT violate; got {C}'
assert A['max_excess'] > 0.01, f'Q4 mutation_A excess too small: {A}'
assert B['max_excess'] > 0.05, f'Q4 mutation_B excess too small: {B}'
print(f"[GATE OK] Q4: A.violations={A['violations']}, B.violations={B['violations']}, C.violations={C['violations']}")


In [ ]:
reflect.log('Q4_mutate',
            'Mutations A (factor) and B (ε²) falsify HR; C (LHS=RHS) is held by the verifier as designed.',
            'HIGH')


## Q5 — Writeup & calibration

Per rubric: at least 5 calibration entries in `writeup.md`. The `reflect.log(...)` calls above (Q1–Q4) cover the assertion gates. This cell records the writeup itself.

In [ ]:
reflect.log('Q5_writeup',
            'Writeup §Q1-§Q4 mirrors the four assertion gates above; calibrations recorded.',
            'MEDIUM')
print('HW1 reflection log so far:')
for r in reflect.dump():
    if r['notebook'].startswith('hw1'):
        print(f"  [{r['level']:>10}] {r['notebook']}/{r['concept']}: {r['claim']}")


**End of HW1.** Five assertion gates fired. Continue to HW2.